In [ ]:
from pathlib import Path
import importlib
import os
import shutil
import subprocess
import sys
import torch

REPO_URL = "https://github.com/ituvtu/reid_investigation.git"
REPO_DIR = Path("/kaggle/working/reid_investigation")


def _run_step(
    step_label: str,
    command: list[str],
    allow_failure: bool = False,
    timeout_seconds: int | None = None,
) -> subprocess.CompletedProcess[str]:
    print(f"[{step_label}] START: {' '.join(command)}")

    try:
        result = subprocess.run(command, capture_output=True, text=True, timeout=timeout_seconds)
    except subprocess.TimeoutExpired as error:
        stdout_text = (error.stdout or "").strip()
        stderr_text = (error.stderr or "").strip()

        if stdout_text:
            print(f"[{step_label}] STDOUT (partial):\n{stdout_text}")
        if stderr_text:
            print(f"[{step_label}] STDERR (partial):\n{stderr_text}")

        timeout_message = f"[{step_label}] TIMED OUT after {timeout_seconds} seconds;"
        if allow_failure:
            print(f"{timeout_message} continuing due to allow_failure=True;")
            return subprocess.CompletedProcess(command, 124, stdout_text, stderr_text)
        raise RuntimeError(timeout_message) from error

    if result.stdout.strip():
        print(f"[{step_label}] STDOUT:\n{result.stdout.strip()}")
    if result.stderr.strip():
        print(f"[{step_label}] STDERR:\n{result.stderr.strip()}")

    if result.returncode != 0 and not allow_failure:
        raise RuntimeError(f"[{step_label}] FAILED with code {result.returncode};")

    if result.returncode == 0:
        print(f"[{step_label}] DONE;")
    else:
        print(f"[{step_label}] FAILED but continuing due to allow_failure=True;")

    return result


def _write_requirements_without_dinov2(source_path: Path, output_path: Path) -> None:
    lines = source_path.read_text(encoding="utf-8").splitlines()
    filtered = [line for line in lines if "dinov2" not in line.lower()]
    output_path.write_text("\n".join(filtered) + "\n", encoding="utf-8")


def _is_build_isolation_issue(text: str) -> bool:
    normalized = text.lower()
    markers = (
        "build backend returned an error",
        "no module named 'numpy'",
        "no module named 'wrapt'",
        "no-build-isolation",
    )
    return any(marker in normalized for marker in markers)


def _is_dinov2_solver_conflict(text: str) -> bool:
    normalized = text.lower()
    return "dinov2" in normalized and "unsatisfiable" in normalized


def _install_requirements_resilient(uv_command: list[str], full_requirements: Path) -> None:
    runtime_requirements = Path("/tmp/requirements_stage2_runtime_no_dinov2.txt")
    _write_requirements_without_dinov2(full_requirements, runtime_requirements)

    include_dinov2_optional = os.environ.get("INSTALL_DINOV2_OPTIONAL", "0") == "1"
    requirements_to_install = full_requirements if include_dinov2_optional else runtime_requirements

    if include_dinov2_optional:
        print("[6/9] INSTALL_DINOV2_OPTIONAL=1; using full requirements;")
    else:
        print("[6/9] Using runtime requirements without dinov2 for compatibility;")

    print("[6/9] Dependency installation can take several minutes;")
    install_result = _run_step(
        "6/9",
        uv_command + ["pip", "install", "-r", str(requirements_to_install), "--system"],
        allow_failure=True,
        timeout_seconds=1800,
    )

    if install_result.returncode == 0:
        return

    merged_output = f"{install_result.stdout}\n{install_result.stderr}"

    if _is_dinov2_solver_conflict(merged_output):
        print("[6/9] dinov2 solver conflict detected; retry without dinov2;")
        requirements_to_install = runtime_requirements
        retry_no_dino = _run_step(
            "6/9A",
            uv_command + ["pip", "install", "-r", str(requirements_to_install), "--system", "--no-build-isolation"],
            allow_failure=True,
            timeout_seconds=1800,
        )
        if retry_no_dino.returncode == 0:
            return
        print("[6/9A] uv retry without dinov2 failed; fallback to pip --no-build-isolation;")
        _run_step(
            "6/9B",
            [sys.executable, "-m", "pip", "install", "-r", str(requirements_to_install), "--no-build-isolation"],
            timeout_seconds=1800,
        )
        return

    if _is_build_isolation_issue(merged_output):
        print("[6/9] Build-isolation issue detected; preinstall build deps and retry;")
        _run_step(
            "6/9X",
            [sys.executable, "-m", "pip", "install", "-q", "numpy", "wrapt", "setuptools", "wheel"],
        )

        retry_result = _run_step(
            "6/9Y",
            uv_command + ["pip", "install", "-r", str(requirements_to_install), "--system", "--no-build-isolation"],
            allow_failure=True,
            timeout_seconds=1800,
        )

        if retry_result.returncode == 0:
            return

        retry_merged_output = f"{retry_result.stdout}\n{retry_result.stderr}"
        if _is_dinov2_solver_conflict(retry_merged_output):
            print("[6/9Y] dinov2 solver conflict after retry; switch to requirements without dinov2;")
            requirements_to_install = runtime_requirements

        print("[6/9Y] uv retry failed; fallback to pip --no-build-isolation (can take several minutes);")
        _run_step(
            "6/9Z",
            [sys.executable, "-m", "pip", "install", "-r", str(requirements_to_install), "--no-build-isolation"],
            timeout_seconds=1800,
        )
        return

    raise RuntimeError(f"[6/9] FAILED with code {install_result.returncode};")


def _normalize_soccernet_root(path: Path) -> Path:
    if path.name.lower() in {"train", "valid", "test", "challenge"}:
        return path.parent
    return path


def _collect_soccernet_candidates(input_root: Path) -> list[Path]:
    candidates: list[Path] = []

    known_candidates = [
        Path("/kaggle/input/soccernet-tracking"),
        Path("/kaggle/input/SoccerNet"),
        Path("/kaggle/input/SoccerNet tracking"),
        Path("/kaggle/input/datasets/theoviel/soccernet-tracking"),
        Path("/kaggle/input/datasets/theoviel/soccernet-tracking/train"),
    ]

    for candidate in known_candidates:
        if candidate.exists() and candidate.is_dir():
            candidates.append(candidate)

    if input_root.exists():
        # Limit scan depth to keep startup fast on large Kaggle mounts.
        for child in sorted(input_root.iterdir()):
            if not child.is_dir():
                continue

            if "soccernet" in child.name.lower():
                candidates.append(child)

            for grandchild in sorted(child.iterdir()):
                if grandchild.is_dir() and "soccernet" in grandchild.name.lower():
                    candidates.append(grandchild)

    unique: list[Path] = []
    seen: set[str] = set()
    for path in candidates:
        normalized = _normalize_soccernet_root(path)
        key = str(normalized.resolve())
        if key in seen:
            continue
        unique.append(normalized)
        seen.add(key)

    return unique


def _list_available_splits(root_path: Path) -> list[str]:
    split_names = {"train", "valid", "test", "challenge"}
    if not root_path.exists() or not root_path.is_dir():
        return []
    return sorted(
        child.name
        for child in root_path.iterdir()
        if child.is_dir() and child.name.lower() in split_names
    )


def _check_torchreid_runtime() -> tuple[bool, str]:
    try:
        importlib.invalidate_caches()
        if "torchreid" in sys.modules:
            del sys.modules["torchreid"]
        torchreid_module = importlib.import_module("torchreid")
    except Exception as error:
        return False, f"import torchreid failed: {error}"

    models_api = getattr(torchreid_module, "models", None)
    if models_api is None or not hasattr(models_api, "build_model"):
        return False, "torchreid.models.build_model is unavailable (incompatible package variant)"

    try:
        from torchreid.utils import load_pretrained_weights as _  # noqa: F401
    except Exception as error:
        return False, f"import torchreid.utils.load_pretrained_weights failed: {error}"

    return True, ""


def _ensure_torchreid_runtime() -> None:
    ok, error_text = _check_torchreid_runtime()
    if ok:
        print("[6/9C] torchreid runtime check: OK;")
        return

    print(f"[6/9C] torchreid runtime check failed: {error_text};")
    print("[6/9C] Applying torchreid fallback installation;")

    _run_step(
        "6/9C",
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "yacs",
            "tensorboard",
            "future",
            "gdown",
        ],
        timeout_seconds=900,
    )
    _run_step(
        "6/9D",
        [sys.executable, "-m", "pip", "uninstall", "-y", "torchreid"],
        allow_failure=True,
        timeout_seconds=600,
    )
    _run_step(
        "6/9E",
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--force-reinstall",
            "--no-deps",
            "git+https://github.com/KaiyangZhou/deep-person-reid.git",
        ],
        timeout_seconds=1200,
    )

    ok_after, error_text_after = _check_torchreid_runtime()
    if not ok_after:
        raise RuntimeError(
            "[6/9E] torchreid is still unavailable after fallback installation; "
            f"details: {error_text_after};"
        )

    print("[6/9E] torchreid runtime check: OK after fallback;")


print("[1/9] Bootstrap started;")
print(f"Current working directory: {Path.cwd()};")
print(f"Repository target path: {REPO_DIR};")

print("[2/9] Repository clone check;")
if not REPO_DIR.exists():
    _run_step("2/9", ["git", "clone", REPO_URL, str(REPO_DIR)])
else:
    print("[2/9] Repository already exists - clone skipped;")

skip_git_pull = os.environ.get("REID_SKIP_GIT_PULL", "0") == "1"
if skip_git_pull:
    print("[2/9A] Git pull skipped because REID_SKIP_GIT_PULL=1;")
else:
    print("[2/9A] Sync repository with remote (git pull --ff-only);")
    pull_result = _run_step(
        "2/9A",
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        allow_failure=True,
        timeout_seconds=300,
    )
    if pull_result.returncode != 0:
        print("[2/9A] Pull failed, continuing with local repository state;")

print("[3/9] Change directory;")
os.chdir(REPO_DIR)
print(f"[3/9] Current working directory: {Path.cwd()};")

print("[4/9] Install uv package manager;")
_run_step("4/9", [sys.executable, "-m", "pip", "install", "-q", "uv"], timeout_seconds=300)

print("[5/9] Resolve uv command;")
uv_cli = shutil.which("uv")
if uv_cli is None:
    uv_command = [sys.executable, "-m", "uv"]
    print("[5/9] Using fallback: python -m uv;")
else:
    uv_command = [uv_cli]
    print(f"[5/9] Using uv executable: {uv_cli};")

print("[6/9] Install Python dependencies from requirements.txt;")
requirements_path = Path("requirements.txt")
_install_requirements_resilient(uv_command, requirements_path)

_ensure_torchreid_runtime()

print("[7/9] Inject repository path into sys.path;")
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"[7/9] Path injected: {str(REPO_DIR) in sys.path};")

print("[8/9] Detect SoccerNet dataset paths (non-recursive scan);")
input_root = Path("/kaggle/input")
soccernet_roots = _collect_soccernet_candidates(input_root)

print(f"[8/9] SoccerNet candidates found: {len(soccernet_roots)};")
if soccernet_roots:
    for path in soccernet_roots[:10]:
        print(f"- {path};")

    selected_root = soccernet_roots[0]
    os.environ.setdefault("SOCCERNET_ROOT_DIR", str(selected_root))
    print(f"[8/9] Selected SOCCERNET_ROOT_DIR: {os.environ['SOCCERNET_ROOT_DIR']};")

    available_splits = _list_available_splits(Path(os.environ["SOCCERNET_ROOT_DIR"]))
    print(f"[8/9] Available splits at selected root: {available_splits};")
else:
    print(f"[8/9] No SoccerNet candidate found under: {input_root};")

os.environ.setdefault("STAGE2_ALLOWED_SPLITS", "train,test")
print(f"[8/9] STAGE2_ALLOWED_SPLITS: {os.environ['STAGE2_ALLOWED_SPLITS']};")

print("[9/9] GPU and environment summary;")
gpu_detected = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if gpu_detected else "No GPU detected"

print(f"GPU detected: {gpu_detected};")
print(f"GPU name: {gpu_name};")
print(f"SOCCERNET_ROOT_DIR: {os.environ.get('SOCCERNET_ROOT_DIR', '<not-set>')};")
print("[9/9] Bootstrap completed;")

In [ ]:
from pathlib import Path
import os

DEFAULT_REPO_URL = "https://github.com/ituvtu/reid_investigation.git"
os.environ.setdefault("REID_REPO_URL", DEFAULT_REPO_URL)

# Optional overrides for Kaggle runtime:
# os.environ["SOCCERNET_ROOT_DIR"] = "/kaggle/input/datasets/theoviel/soccernet-tracking"
# os.environ["SOCCERNET_PASSWORD"] = "<password_if_required>"

print("Working directory:", Path.cwd())
print("models exists:", os.path.exists("models"))
print("configs exists:", os.path.exists("configs"))
print("REID_REPO_URL:", os.environ.get("REID_REPO_URL", ""))

In [ ]:
from pathlib import Path
import importlib
import subprocess
import sys
import warnings


def _run_or_fail(command: list[str]) -> None:
    print("$", " ".join(command))
    result = subprocess.run(command, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    if result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(command)}")


def _check_torchreid_runtime() -> tuple[bool, str]:
    try:
        importlib.invalidate_caches()
        if "torchreid" in sys.modules:
            del sys.modules["torchreid"]
        torchreid_module = importlib.import_module("torchreid")
    except Exception as error:
        return False, f"import torchreid failed: {error}"

    models_api = getattr(torchreid_module, "models", None)
    if models_api is None or not hasattr(models_api, "build_model"):
        return False, "torchreid.models.build_model is unavailable"

    try:
        from torchreid.utils import load_pretrained_weights as _  # noqa: F401
    except Exception as error:
        return False, f"import torchreid.utils.load_pretrained_weights failed: {error}"

    return True, ""


runtime_ok, runtime_error = _check_torchreid_runtime()
if not runtime_ok:
    print(f"torchreid runtime mismatch detected: {runtime_error};")
    print("Applying fallback repair for deep-person-reid;")
    _run_or_fail([sys.executable, "-m", "pip", "install", "yacs", "tensorboard", "future", "gdown"])
    _run_or_fail([sys.executable, "-m", "pip", "uninstall", "-y", "torchreid"])
    _run_or_fail(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--force-reinstall",
            "--no-deps",
            "git+https://github.com/KaiyangZhou/deep-person-reid.git",
        ]
    )
    runtime_ok, runtime_error = _check_torchreid_runtime()
    if not runtime_ok:
        raise RuntimeError(f"torchreid runtime is still invalid after repair: {runtime_error};")

print("torchreid runtime check: OK;")

from models.detectors import YOLODetector
from models.reid import OSNetReID
from models.trackers import ByteTrackTracker
from utils import load_stage2_reid_config, stage2_component_mappings

config = load_stage2_reid_config("configs/stage2_reid.yaml")
detector_config, tracker_config, reid_config = stage2_component_mappings(config)

requested_weights = str(detector_config.get("weights_path", "")).strip()
requested_model = str(detector_config.get("model_name", "")).strip()
weights_path = Path(requested_weights) if requested_weights else None

if requested_model.lower().startswith("yolo26"):
    detector_config["weights_path"] = requested_model
    print(f"YOLOv26 library mode enabled: source={requested_model};")
elif requested_weights and weights_path is not None and not weights_path.exists():
    raise FileNotFoundError(
        f"Weights file not found: {requested_weights}; provide a valid path or use a model name source."
    )

detector = YOLODetector.from_config(detector_config)
detector.load()
detector.warmup(image_size=(640, 640), runs=1)

shared_device = detector.device
tracker_config["device"] = shared_device
reid_config["device"] = shared_device

tracker = ByteTrackTracker.from_config(tracker_config)
tracker.initialize()

reid_model = OSNetReID.from_config(reid_config)
with warnings.catch_warnings():
    warnings.filterwarnings('ignore', category=FutureWarning, message='.*weights_only=False.*')
    reid_model.load()

print(f"Detector device: {detector.device}")
print(f"Tracker device: {tracker.device}")
print(f"ReID device: {reid_model.device}")
print(f"Experiment: {config.experiment_name}")

if not str(detector.device).startswith("cuda"):
    print("Warning: CUDA was not found - running on CPU;")

In [ ]:
import csv
import os
from pathlib import Path

import cv2
from core.base_tracker import Track
from utils import SoccerNetLoader

if config.dataset is None or config.dataset.soccernet is None:
    raise ValueError("Missing dataset.soccernet configuration in configs/stage2_reid.yaml")

soccernet_cfg = config.dataset.soccernet
override_root = os.environ.get("SOCCERNET_ROOT_DIR", "").strip()
override_password = os.environ.get("SOCCERNET_PASSWORD", "").strip() or soccernet_cfg.password

candidate_roots = [
    Path(override_root).expanduser() if override_root else None,
    Path(soccernet_cfg.root_dir).expanduser(),
    Path('/kaggle/input/SoccerNet tracking'),
    Path('/kaggle/input/soccernet-tracking'),
    Path('/kaggle/input/datasets/theoviel/soccernet-tracking'),
]

print('Dataset root candidates (in priority order):')
for index, candidate in enumerate(candidate_roots, start=1):
    if candidate is None:
        print(f"{index}. <empty override>;")
        continue
    exists_text = 'exists' if candidate.exists() else 'missing'
    print(f"{index}. {candidate} [{exists_text}];")

resolved_root: Path | None = None
for candidate in candidate_roots:
    if candidate is not None and candidate.exists():
        resolved_root = candidate
        break

if resolved_root is None:
    rendered_candidates = [str(path) for path in candidate_roots if path is not None]
    raise RuntimeError(
        'SoccerNet root directory was not found. Checked candidates: ' + ', '.join(rendered_candidates) + ';'
    )

target_root = resolved_root
dataset_mapping = {
    'root_dir': str(target_root),
    'subset': soccernet_cfg.subset,
    'split': list(soccernet_cfg.split),
    'password': override_password,
    'auto_download': False,
}

print('Dataset configuration:')
print(f"root_dir: {dataset_mapping['root_dir']};")
print(f"subset: {dataset_mapping['subset']};")
print(f"split from config: {dataset_mapping['split']};")
print('auto_download: False (Kaggle dataset already extracted);')

soccernet_loader = SoccerNetLoader.from_config(dataset_mapping)
dataset_root = target_root


def _list_split_directories(root: Path) -> dict[str, Path]:
    known_splits = ('train', 'test', 'challenge', 'valid')
    split_dirs: dict[str, Path] = {}
    for split_name in known_splits:
        split_path = root / split_name
        if split_path.exists() and split_path.is_dir():
            split_dirs[split_name] = split_path
    return split_dirs


def _count_sequences_in_split(split_path: Path) -> int:
    sequence_count = 0
    for sequence_dir in sorted(split_path.iterdir()):
        if sequence_dir.is_dir() and (sequence_dir / 'img1').is_dir():
            sequence_count += 1
    return sequence_count


def _find_video_files(root: Path) -> list[Path]:
    extensions = ('.mp4', '.avi', '.mov', '.mkv')
    return sorted([path for path in root.rglob('*') if path.is_file() and path.suffix.lower() in extensions])


def _find_annotation_files(root: Path) -> tuple[list[Path], list[Path], list[Path]]:
    json_files = sorted(root.rglob('*.json'))
    csv_files = sorted(root.rglob('*.csv'))
    txt_files = sorted(root.rglob('*.txt'))
    return json_files, csv_files, txt_files


def _load_mot_table_tracks(path: Path) -> dict[int, list[Track]]:
    tracks_by_frame: dict[int, list[Track]] = {}
    with path.open('r', encoding='utf-8', newline='') as file:
        reader = csv.reader(file)
        for row in reader:
            if len(row) < 6:
                continue
            try:
                frame_id = int(float(row[0]))
                track_id = int(float(row[1]))
                x = float(row[2])
                y = float(row[3])
                w = float(row[4])
                h = float(row[5])
                confidence = float(row[6]) if len(row) > 6 and row[6] != '' else 1.0
            except Exception:
                continue

            tracks_by_frame.setdefault(frame_id, []).append(
                Track(
                    track_id=track_id,
                    bbox_xyxy=(x, y, x + w, y + h),
                    confidence=confidence,
                    class_id=0,
                    embedding=None,
                )
            )
    return tracks_by_frame


def iter_image_sequence_frames(image_files: list[Path], max_frames: int | None = None, stride: int = 1):
    produced = 0
    for image_path in image_files:
        frame_id = None
        try:
            frame_id = int(image_path.stem)
        except Exception:
            continue

        if frame_id % stride != 0:
            continue

        frame = cv2.imread(str(image_path))
        if frame is None:
            continue

        yield frame_id, frame
        produced += 1

        if max_frames is not None and produced >= max_frames:
            break


def _choose_sequence_annotation(sequence_root: Path) -> Path | None:
    gt_dir = sequence_root / 'gt'
    candidates: list[Path] = []

    if gt_dir.exists():
        candidates.extend(sorted(gt_dir.glob('*.json')))
        candidates.extend(sorted(gt_dir.glob('*.csv')))
        candidates.extend(sorted(gt_dir.glob('*.txt')))

    candidates.extend(sorted(sequence_root.glob('*.json')))
    candidates.extend(sorted(sequence_root.glob('*.csv')))
    candidates.extend(sorted(sequence_root.glob('*.txt')))

    filtered_candidates = [
        path for path in candidates
        if path.is_file() and ('gt' in path.name.lower() or 'tracking' in path.name.lower() or path.suffix.lower() == '.json')
    ]

    if filtered_candidates:
        return filtered_candidates[0]
    return candidates[0] if candidates else None


def _load_annotation_tracks(annotation_path: Path) -> dict[int, list[Track]]:
    if annotation_path.suffix.lower() == '.json':
        annotation_payload = soccernet_loader.load_tracking_annotations(annotation_path=annotation_path)
        return soccernet_loader.map_tracking_annotations_to_tracks(annotation_payload)

    if annotation_path.suffix.lower() in {'.csv', '.txt'}:
        return _load_mot_table_tracks(annotation_path)

    raise RuntimeError(f'Unsupported annotation format: {annotation_path.suffix}')


def _discover_sequence_sources(root: Path) -> list[dict]:
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
    img_dirs = sorted([path for path in root.rglob('img1') if path.is_dir()])
    discovered_sources: list[dict] = []

    for img_dir in img_dirs:
        sequence_root = img_dir.parent
        image_files = sorted(
            [path for path in img_dir.iterdir() if path.is_file() and path.suffix.lower() in image_extensions],
            key=lambda path: path.name,
        )
        if not image_files:
            continue

        annotation_path = _choose_sequence_annotation(sequence_root)
        if annotation_path is None:
            continue

        gt_tracks = _load_annotation_tracks(annotation_path)
        if not gt_tracks:
            continue

        try:
            source_name = sequence_root.relative_to(root).as_posix()
        except Exception:
            source_name = sequence_root.name

        discovered_sources.append(
            {
                'source_name': source_name,
                'frame_source_kind': 'image_sequence',
                'source_path': img_dir,
                'image_files': image_files,
                'annotation_path': annotation_path,
                'gt_tracks_by_frame': gt_tracks,
            }
        )

    return discovered_sources


def _split_name_from_source(source_name: str) -> str:
    source_parts = [part.lower() for part in Path(source_name).parts]
    for split_name in ('train', 'test', 'challenge', 'valid'):
        if split_name in source_parts:
            return split_name
    return 'unknown'


split_directories = _list_split_directories(dataset_root)
if not split_directories:
    raise RuntimeError(f'No split directories found under dataset root: {dataset_root};')

print('Detected split directories and sequence counts:')
for split_name, split_path in split_directories.items():
    sequence_count = _count_sequences_in_split(split_path)
    print(f"- {split_name}: {split_path} | sequences with img1={sequence_count};")

video_files = _find_video_files(dataset_root)
json_files, csv_files, txt_files = _find_annotation_files(dataset_root)
print(f'Videos found: {len(video_files)};')
print(f'JSON annotations found: {len(json_files)};')
print(f'CSV annotations found: {len(csv_files)};')
print(f'TXT annotations found: {len(txt_files)};')

requested_source_count = int(os.environ.get('STAGE2_SOURCE_COUNT', '10'))
if requested_source_count < 1:
    raise ValueError('STAGE2_SOURCE_COUNT must be >= 1;')

allowed_splits_env = os.environ.get('STAGE2_ALLOWED_SPLITS', 'train,test')
allowed_splits = {item.strip().lower() for item in allowed_splits_env.split(',') if item.strip()}
if not allowed_splits:
    raise ValueError('STAGE2_ALLOWED_SPLITS must define at least one split;')

print(f"Requested source count: {requested_source_count};")
print(f"Allowed splits: {sorted(allowed_splits)};")

available_sequence_sources = _discover_sequence_sources(dataset_root)
if not available_sequence_sources:
    raise RuntimeError('No valid tracking sequences with img1 and annotations were found;')

split_totals: dict[str, int] = {}
for source in available_sequence_sources:
    split_name = _split_name_from_source(str(source['source_name']))
    split_totals[split_name] = split_totals.get(split_name, 0) + 1

print(f"Discovered sequence sources before split filtering: {len(available_sequence_sources)};")
for split_name in sorted(split_totals):
    print(f"- discovered in {split_name}: {split_totals[split_name]};")

filtered_sequence_sources: list[dict] = []
for source in available_sequence_sources:
    source_parts = {part.lower() for part in Path(str(source['source_name'])).parts}
    if source_parts & allowed_splits:
        filtered_sequence_sources.append(source)

removed_count = len(available_sequence_sources) - len(filtered_sequence_sources)
available_sequence_sources = filtered_sequence_sources
print(f"Sources after split filtering: {len(available_sequence_sources)}; removed={removed_count};")

if not available_sequence_sources:
    raise RuntimeError('No valid tracking sequences were found for the requested splits;')

evaluation_sources = available_sequence_sources[:requested_source_count]
if len(evaluation_sources) < requested_source_count:
    raise RuntimeError(
        f"Requested {requested_source_count} sources but only found {len(evaluation_sources)} valid sequences;"
    )

# Compatibility variables for downstream cells and ad-hoc checks.
primary_source = evaluation_sources[0]
frame_source_kind = primary_source['frame_source_kind']
sequence_image_files = primary_source['image_files']
video_path = primary_source['source_path']
annotation_path = primary_source['annotation_path']
gt_tracks_by_frame = primary_source['gt_tracks_by_frame']

print(f'Selected sources for evaluation: {len(evaluation_sources)};')
for index, source in enumerate(evaluation_sources, start=1):
    gt_frames = len(source['gt_tracks_by_frame'])
    image_count = len(source['image_files'])
    split_name = _split_name_from_source(str(source['source_name']))
    print(
        f"{index}. split={split_name} | {source['source_name']} | images={image_count} | "
        f"gt_frames={gt_frames} | ann={source['annotation_path']};"
    )

In [ ]:
import importlib
import utils.metrics as metrics_module

importlib.reload(metrics_module)
from utils import compute_mot_id_metrics

print('Reloaded utils.metrics with NumPy 2 compatibility shim;')

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt

from core.base_tracker import Track
from utils import compute_mot_id_metrics

# Add compatibility for NumPy 2.x;
if not hasattr(np, 'asfarray'):
    def _np_asfarray_compat(values, dtype=float):
        return np.asarray(values, dtype=dtype)

    np.asfarray = _np_asfarray_compat

try:
    import cv2
except Exception:
    cv2 = None
    print('Warning: OpenCV is unavailable - visualization will be limited;')

if 'evaluation_sources' not in globals() or not evaluation_sources:
    raise RuntimeError('evaluation_sources is missing - run Cell 4 first;')

if 'reid_model' not in globals():
    raise RuntimeError('reid_model is missing - run Cell 3 first;')

if 'tracker' not in globals():
    raise RuntimeError('tracker is missing - run Cell 3 first;')

max_frames_per_source = int(os.environ.get('STAGE2_MAX_FRAMES', '120'))
frame_stride = int(os.environ.get('STAGE2_FRAME_STRIDE', '2'))
iou_threshold = float(os.environ.get('STAGE2_IOU_THRESHOLD', '0.5'))

if max_frames_per_source < 1:
    raise ValueError('STAGE2_MAX_FRAMES must be >= 1;')
if frame_stride < 1:
    raise ValueError('STAGE2_FRAME_STRIDE must be >= 1;')

association_alpha_value = float(getattr(tracker, 'association_alpha', tracker_config.get('association_alpha', 0.5)))
default_input_size = reid_config.get('input_size', [256, 128])
reid_input_height = int(getattr(reid_model, 'input_height', int(default_input_size[0])))
reid_input_width = int(getattr(reid_model, 'input_width', int(default_input_size[1])))

print(f'Current association alpha parameter: {association_alpha_value:.3f};')
print(f'ReID crop size: {reid_input_height}x{reid_input_width};')


def _offset_tracks_by_frame_and_id(
    tracks_by_frame: dict[int, list[Track]],
    frame_offset: int,
    track_offset: int,
) -> dict[int, list[Track]]:
    remapped: dict[int, list[Track]] = {}
    for frame_id, tracks in tracks_by_frame.items():
        remapped_frame = int(frame_id) + frame_offset
        remapped[remapped_frame] = [
            Track(
                track_id=int(track.track_id) + track_offset,
                bbox_xyxy=track.bbox_xyxy,
                confidence=float(track.confidence),
                class_id=int(track.class_id),
                embedding=track.embedding,
            )
            for track in tracks
        ]
    return remapped


def _extract_crops_for_reid(frame: np.ndarray, detections) -> list[np.ndarray]:
    # Clip bounding boxes to frame boundaries and resize crops to the OSNet input size;
    if frame is None or frame.size == 0:
        return [np.zeros((reid_input_height, reid_input_width, 3), dtype=np.uint8) for _ in detections]

    frame_height, frame_width = frame.shape[:2]
    if frame_height < 1 or frame_width < 1:
        return [np.zeros((reid_input_height, reid_input_width, 3), dtype=np.uint8) for _ in detections]

    crops: list[np.ndarray] = []
    for detection in detections:
        x1, y1, x2, y2 = detection.bbox_xyxy

        left = int(np.clip(np.floor(x1), 0, frame_width - 1))
        top = int(np.clip(np.floor(y1), 0, frame_height - 1))
        right = int(np.clip(np.ceil(x2), left + 1, frame_width))
        bottom = int(np.clip(np.ceil(y2), top + 1, frame_height))

        crop = frame[top:bottom, left:right]
        if cv2 is None or crop.size == 0:
            resized_crop = np.zeros((reid_input_height, reid_input_width, 3), dtype=np.uint8)
        else:
            resized_crop = cv2.resize(
                crop,
                (reid_input_width, reid_input_height),
                interpolation=cv2.INTER_LINEAR,
            )

        crops.append(resized_crop)

    return crops


preview_frames: list[tuple[str, int, np.ndarray]] = []
per_source_metrics_state: list[dict] = []
combined_ground_truth: dict[int, list[Track]] = {}
combined_predictions: dict[int, list[Track]] = {}

for source_index, source in enumerate(evaluation_sources):
    source_name = str(source['source_name'])
    source_kind = str(source['frame_source_kind'])
    source_gt_tracks_by_frame = source['gt_tracks_by_frame']

    if source_kind == 'video':
        frame_iterator = soccernet_loader.iter_video_frames(
            video_path=source['source_path'],
            max_frames=max_frames_per_source,
            stride=frame_stride,
        )
    else:
        frame_iterator = iter_image_sequence_frames(
            image_files=source['image_files'],
            max_frames=max_frames_per_source,
            stride=frame_stride,
        )

    tracker.initialize()

    source_latencies: list[float] = []
    source_predictions_by_frame: dict[int, list[Track]] = {}

    for frame_index, frame in frame_iterator:
        frame_start = time.perf_counter()
        detections = detector.predict(frame)

        if detections:
            crops = _extract_crops_for_reid(frame, detections)
            embeddings = reid_model.extract(crops)
        else:
            embeddings = None

        tracks = tracker.update(detections=detections, embeddings=embeddings, frame_index=frame_index)
        source_latencies.append(time.perf_counter() - frame_start)
        source_predictions_by_frame[frame_index] = tracks

        if len(preview_frames) < 3:
            vis_frame = frame.copy()
            if cv2 is not None:
                cv2.putText(
                    vis_frame,
                    f'alpha={association_alpha_value:.2f}',
                    (12, 24),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0, 255, 255),
                    2,
                    cv2.LINE_AA,
                )
                for track in tracks:
                    x1, y1, x2, y2 = [int(value) for value in track.bbox_xyxy]
                    cv2.rectangle(vis_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(
                        vis_frame,
                        f'ID {track.track_id}',
                        (x1, max(44, y1 - 8)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (255, 255, 255),
                        2,
                        cv2.LINE_AA,
                    )
            preview_frames.append((source_name, int(frame_index), vis_frame))

    if not source_predictions_by_frame:
        raise RuntimeError(f'No frames were processed for source: {source_name};')

    source_metric_values = compute_mot_id_metrics(
        ground_truth_by_frame=source_gt_tracks_by_frame,
        predictions_by_frame=source_predictions_by_frame,
        iou_threshold=iou_threshold,
    )

    source_fps = (1.0 / float(np.mean(source_latencies))) if source_latencies else 0.0
    processed_frames = len(source_predictions_by_frame)

    per_source_metrics_state.append(
        {
            'Source': source_name,
            'MOTA': float(source_metric_values['MOTA']),
            'IDF1': float(source_metric_values['IDF1']),
            'ID Swaps': int(source_metric_values['ID Swaps']),
            'FPS': float(source_fps),
            'Processed Frames': int(processed_frames),
            'Association Alpha': float(association_alpha_value),
        }
    )

    # Isolate ID spaces across sources;
    frame_offset = (source_index + 1) * 1_000_000
    track_offset = (source_index + 1) * 100_000

    combined_ground_truth.update(
        _offset_tracks_by_frame_and_id(source_gt_tracks_by_frame, frame_offset=frame_offset, track_offset=track_offset)
    )
    combined_predictions.update(
        _offset_tracks_by_frame_and_id(source_predictions_by_frame, frame_offset=frame_offset, track_offset=track_offset)
    )

if len(per_source_metrics_state) != len(evaluation_sources):
    raise RuntimeError('Not all selected sources produced valid metrics;')

aggregate_metric_values = compute_mot_id_metrics(
    ground_truth_by_frame=combined_ground_truth,
    predictions_by_frame=combined_predictions,
    iou_threshold=iou_threshold,
)

average_fps = float(np.mean([item['FPS'] for item in per_source_metrics_state])) if per_source_metrics_state else 0.0

metrics_state = {
    'MOTA': float(aggregate_metric_values['MOTA']),
    'IDF1': float(aggregate_metric_values['IDF1']),
    'ID Swaps': int(aggregate_metric_values['ID Swaps']),
    'FPS': float(average_fps),
    'Sources Evaluated': int(len(per_source_metrics_state)),
    'Association Alpha': float(association_alpha_value),
}

preview_count = len(preview_frames)
if preview_count > 0:
    fig, axes = plt.subplots(1, preview_count, figsize=(5 * preview_count, 4))
    if preview_count == 1:
        axes = [axes]

    for index in range(preview_count):
        source_name, frame_index, frame_to_show = preview_frames[index]
        if cv2 is not None:
            frame_to_show = cv2.cvtColor(frame_to_show, cv2.COLOR_BGR2RGB)
        axes[index].imshow(frame_to_show)
        axes[index].set_title(f'{source_name} - frame {frame_index} - alpha {association_alpha_value:.2f}')
        axes[index].axis('off')

    plt.tight_layout()

print('Status: per-source and aggregate metric computation completed;')
print(f"Sources evaluated: {metrics_state['Sources Evaluated']};")

In [ ]:
from pathlib import Path

import pandas as pd

if 'metrics_state' not in globals():
    raise RuntimeError('metrics_state is missing - run Cell 6 first;')
if 'per_source_metrics_state' not in globals():
    raise RuntimeError('per_source_metrics_state is missing - run Cell 6 first;')

aggregate_rows = [
    {'Metric': 'MOTA', 'Value': float(metrics_state['MOTA'])},
    {'Metric': 'IDF1', 'Value': float(metrics_state['IDF1'])},
    {'Metric': 'ID Swaps', 'Value': int(metrics_state['ID Swaps'])},
    {'Metric': 'FPS (avg across sources)', 'Value': float(metrics_state['FPS'])},
    {'Metric': 'Sources Evaluated', 'Value': int(metrics_state['Sources Evaluated'])},
    {'Metric': 'Association Alpha', 'Value': float(metrics_state['Association Alpha'])},
]

aggregate_metrics_table = pd.DataFrame(aggregate_rows)
per_source_metrics_table = pd.DataFrame(per_source_metrics_state)

aggregate_display = aggregate_metrics_table.copy()
aggregate_display['Value'] = aggregate_display['Value'].apply(
    lambda value: f'{value:.3f}' if isinstance(value, float) else value
)

per_source_display = per_source_metrics_table.copy()
for column in ['MOTA', 'IDF1', 'FPS', 'Association Alpha']:
    if column in per_source_display.columns:
        per_source_display[column] = per_source_display[column].map(lambda value: f'{float(value):.3f}')

print('Metrics: aggregate summary across selected sources;')
display(aggregate_display)

print('Metrics: per-source breakdown;')
display(per_source_display)

outputs_dir = Path('outputs')
outputs_dir.mkdir(parents=True, exist_ok=True)

aggregate_metrics_path = outputs_dir / 'stage2_metrics.csv'
per_source_metrics_path = outputs_dir / 'stage2_metrics_by_source.csv'

aggregate_metrics_table.to_csv(aggregate_metrics_path, index=False)
per_source_metrics_table.to_csv(per_source_metrics_path, index=False)

print(f'Saved aggregate metrics CSV: {aggregate_metrics_path};')
print(f'Saved per-source metrics CSV: {per_source_metrics_path};')